In [1]:
import cv2
import numpy as np
import imutils
from google.colab.patches import cv2_imshow
import time

class AdvancedMotionDetector:
    def __init__(self, frame_size=(800, 600), min_contour_area=100, blur_ksize=(21, 21), dilate_iterations=7):
        self.frame_size = frame_size
        self.min_contour_area = min_contour_area
        self.blur_ksize = blur_ksize
        self.dilate_iterations = dilate_iterations
        self.backSub = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=120, detectShadows=True)

    def process_video(self, video_path, output_filename=None):
        """
        Memproses file video untuk mendeteksi gerakan dan menyimpannya jika nama file output diberikan.

        Args:
            video_path (str): Path ke file video input.
            output_filename (str, optional): Path untuk menyimpan video hasil.
                                              Jika None, video tidak akan disimpan.
        """
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"Error: Tidak bisa membuka video di {video_path}")
            return

        writer = None

        if output_filename:
            fps = cap.get(cv2.CAP_PROP_FPS)
            frame_width, frame_height = self.frame_size

            fourcc = cv2.VideoWriter_fourcc(*'mp4v')

            writer = cv2.VideoWriter(output_filename, fourcc, fps, (frame_width, frame_height))

            if not writer.isOpened():
                print(f"Error: Gagal membuat file video di {output_filename}")
                cap.release()
                return

            print(f"Video sedang diproses dan akan disimpan ke {output_filename}...")

        while True:
            success, frame = cap.read()
            if not success:
                break

            frame = cv2.resize(frame, self.frame_size)
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            gray_blurred = cv2.GaussianBlur(gray, self.blur_ksize, 0)
            fg_mask = self.backSub.apply(gray_blurred)
            fg_mask = cv2.erode(fg_mask, None, iterations=1)
            fg_mask = cv2.dilate(fg_mask, None, iterations=4)

            cnts = cv2.findContours(fg_mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cnts = imutils.grab_contours(cnts)

            for c in cnts:
                if cv2.contourArea(c) < self.min_contour_area:
                    continue
                (x, y, w, h) = cv2.boundingRect(c)
                cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

            if writer:
                writer.write(frame)

            cv2_imshow(frame)

        print("Proses selesai.")
        cap.release()
        if writer:
            writer.release()
            print(f"Video berhasil disimpan di {output_filename}")

    def process_realtime_stream(self, stream_url, record_on_motion=True, record_timeout=5.0):
        """
        Memproses stream CCTV real-time, mendeteksi gerakan, dan merekam klip saat gerakan terdeteksi.

        Args:
            stream_url (str or int): URL stream RTSP/HTTP atau ID webcam (misal: 0).
            record_on_motion (bool): Jika True, akan menyimpan klip video saat ada gerakan.
            record_timeout (float): Detik tanpa gerakan sebelum berhenti merekam.
        """
        print("Mencoba membuka stream...")
        cap = cv2.VideoCapture(stream_url)

        if not cap.isOpened():
            print(f"Error: Tidak bisa membuka stream di {stream_url}")
            return

        print("Stream berhasil dibuka. Tekan 'q' pada jendela output untuk berhenti.")

        writer = None
        is_recording = False
        last_motion_time = 0

        while True:
            success, frame = cap.read()
            if not success:
                print("Stream terputus atau berakhir. Mencoba menyambung kembali...")
                # Coba buka kembali stream
                cap.release()
                time.sleep(5) # Beri jeda sebelum mencoba lagi
                cap = cv2.VideoCapture(stream_url)
                if not cap.isOpened():
                    print("Gagal menyambung kembali. Keluar.")
                    break
                continue

            frame = cv2.resize(frame, self.frame_size)
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            gray_blurred = cv2.GaussianBlur(gray, self.blur_ksize, 0)
            fg_mask = self.backSub.apply(gray_blurred)
            fg_mask = cv2.erode(fg_mask, None, iterations=1)
            fg_mask = cv2.dilate(fg_mask, None, iterations=self.dilate_iterations)

            cnts = cv2.findContours(fg_mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cnts = imutils.grab_contours(cnts)

            motion_detected = False
            for c in cnts:
                if cv2.contourArea(c) < self.min_contour_area:
                    continue

                motion_detected = True
                (x, y, w, h) = cv2.boundingRect(c)
                cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 2) # Merah untuk real-time

            # --- Logika Smart Recording ---
            if record_on_motion:
                if motion_detected:
                    last_motion_time = time.time()
                    if not is_recording:
                        is_recording = True
                        # Buat nama file unik berdasarkan timestamp
                        output_filename = f"motion_{time.strftime('%Y%m%d_%H%M%S')}.avi"
                        fourcc = cv2.VideoWriter_fourcc(*'XVID')
                        writer = cv2.VideoWriter(output_filename, fourcc, 20.0, self.frame_size) # Asumsikan FPS 20
                        print(f"MOTION DETECTED! Mulai merekam ke {output_filename}")

                elif is_recording and (time.time() - last_motion_time) > record_timeout:
                    is_recording = False
                    writer.release()
                    writer = None
                    print(f"Rekaman berhenti. File disimpan.")

            if is_recording:
                # Tambahkan indikator visual bahwa sedang merekam
                cv2.circle(frame, (30, 30), 10, (0, 0, 255), -1)
                cv2.putText(frame, "REC", (50, 37), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                writer.write(frame)

            # Tampilkan frame di Colab
            cv2_imshow(frame)

            # Tunggu input keyboard untuk keluar. `waitKey` sangat penting untuk `imshow` bisa refresh.
            # Di lingkungan non-Colab, `cv2.waitKey(1)` sudah cukup.
            # Di Colab, `imshow` menangani ini, jadi kita hanya butuh cara untuk break.
            # Sayangnya, cara `waitKey` standar tidak bekerja baik dengan `cv2_imshow`.
            # Loop ini akan berjalan terus sampai Anda menghentikannya secara manual di Colab.

        print("Proses dihentikan.")
        cap.release()
        if writer is not None:
            writer.release()
            print("Rekaman terakhir disimpan.")

In [4]:
motion = AdvancedMotionDetector(frame_size=(800, 600), min_contour_area=120)
motion.process_video("test.mp4", output_filename='annotatedvideo1.mp4')

Output hidden; open in https://colab.research.google.com to view.

In [3]:
detector = AdvancedMotionDetector(
    frame_size=(800, 600),
    min_contour_area=500,
    dilate_iterations=10
)
cctv_url = 'http://79.8.16.194/cgi-bin/faststream.jpg?stream=half&fps=15&rand=COUNTER'
detector.process_realtime_stream(cctv_url, record_on_motion=True, record_timeout=0.5)

Output hidden; open in https://colab.research.google.com to view.